In [ ]:
# Tarea 3

In [ ]:
'''
Usar dataframes de pyspark para manipular datos
Realizar manipulación de filas y columnas en los datos elegidos
Modificar datos
Agregar nuevas columnas calculadas
Filtrar resultados
'''

In [2]:
!pip install pyspark

In [6]:
# definición del dataset por medio de pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("dataset") \
    .getOrCreate()

In [5]:
#se carga el dataset
df = spark.read.csv(
    "dataset.csv",
    header=True,
    multiLine=True,
    quote='"',
    escape='"',
    inferSchema=False
)

df.show(5)

+---+--------------------+--------------------+--------------------+--------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|_c0|            track_id|             artists|          album_name|          track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|  tempo|time_signature|track_genre|
+---+--------------------+--------------------+--------------------+--------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|  0|5SuOikwiRyPMVoIQD...|         Gen Hoshino|              Comedy|              Comedy|        73|     230666|   False|       0.676| 0.461|  1|  -6.746|   0|      0.143|      0.0322|        1.01e-06|   0.358|  0.715| 87.917|            

In [7]:
# visualziacion de la estrucutra del dataset (algo similar a la tarea anterior)
df.printSchema()

df.select(
    "track_name",
    "artists",
    "track_genre",
    "popularity"
).show(10, truncate=False)

root
 |-- _c0: string (nullable = true)
 |-- track_id: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- album_name: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- popularity: string (nullable = true)
 |-- duration_ms: string (nullable = true)
 |-- explicit: string (nullable = true)
 |-- danceability: string (nullable = true)
 |-- energy: string (nullable = true)
 |-- key: string (nullable = true)
 |-- loudness: string (nullable = true)
 |-- mode: string (nullable = true)
 |-- speechiness: string (nullable = true)
 |-- acousticness: string (nullable = true)
 |-- instrumentalness: string (nullable = true)
 |-- liveness: string (nullable = true)
 |-- valence: string (nullable = true)
 |-- tempo: string (nullable = true)
 |-- time_signature: string (nullable = true)
 |-- track_genre: string (nullable = true)

+--------------------------+------------------------------------+-----------+----------+
|track_name                |artists              

In [8]:
# columnas manipuladas
df_select = df.select(
    "track_name",
    "artists",
    "popularity",
    "danceability",
    "energy",
    "tempo",
    "track_genre"
)

df_select.show(10)

+--------------------+--------------------+----------+------------+------+-------+-----------+
|          track_name|             artists|popularity|danceability|energy|  tempo|track_genre|
+--------------------+--------------------+----------+------------+------+-------+-----------+
|              Comedy|         Gen Hoshino|        73|       0.676| 0.461| 87.917|   acoustic|
|    Ghost - Acoustic|        Ben Woodward|        55|        0.42| 0.166| 77.489|   acoustic|
|      To Begin Again|Ingrid Michaelson...|        57|       0.438| 0.359| 76.332|   acoustic|
|Can't Help Fallin...|        Kina Grannis|        71|       0.266|0.0596| 181.74|   acoustic|
|             Hold On|    Chord Overstreet|        82|       0.618| 0.443|119.949|   acoustic|
|Days I Will Remember|        Tyrone Wells|        58|       0.688| 0.481| 98.017|   acoustic|
|       Say Something|A Great Big World...|        74|       0.407| 0.147|141.284|   acoustic|
|           I'm Yours|          Jason Mraz|       

In [9]:
'''
por ejemplo renombramos
track name a que pase a ser solo song
'''
df_select = df_select.withColumnRenamed(
    "track_name",
    "song"
)

df_select.show(5)

+--------------------+--------------------+----------+------------+------+-------+-----------+
|                song|             artists|popularity|danceability|energy|  tempo|track_genre|
+--------------------+--------------------+----------+------------+------+-------+-----------+
|              Comedy|         Gen Hoshino|        73|       0.676| 0.461| 87.917|   acoustic|
|    Ghost - Acoustic|        Ben Woodward|        55|        0.42| 0.166| 77.489|   acoustic|
|      To Begin Again|Ingrid Michaelson...|        57|       0.438| 0.359| 76.332|   acoustic|
|Can't Help Fallin...|        Kina Grannis|        71|       0.266|0.0596| 181.74|   acoustic|
|             Hold On|    Chord Overstreet|        82|       0.618| 0.443|119.949|   acoustic|
+--------------------+--------------------+----------+------------+------+-------+-----------+
only showing top 5 rows


In [10]:
# convertimos las variables a tipo numérico
from pyspark.sql.functions import expr

numeric_cols = [
    "popularity",
    "danceability",
    "energy",
    "tempo"
]

for c in numeric_cols:
    df_select = df_select.withColumn(
        c,
        expr(f"try_cast({c} as double)")
    )

In [11]:
# se eliminan valores nulos
df_select = df_select.dropna()

df_select.show(10)

+--------------------+--------------------+----------+------------+------+-------+-----------+
|                song|             artists|popularity|danceability|energy|  tempo|track_genre|
+--------------------+--------------------+----------+------------+------+-------+-----------+
|              Comedy|         Gen Hoshino|      73.0|       0.676| 0.461| 87.917|   acoustic|
|    Ghost - Acoustic|        Ben Woodward|      55.0|        0.42| 0.166| 77.489|   acoustic|
|      To Begin Again|Ingrid Michaelson...|      57.0|       0.438| 0.359| 76.332|   acoustic|
|Can't Help Fallin...|        Kina Grannis|      71.0|       0.266|0.0596| 181.74|   acoustic|
|             Hold On|    Chord Overstreet|      82.0|       0.618| 0.443|119.949|   acoustic|
|Days I Will Remember|        Tyrone Wells|      58.0|       0.688| 0.481| 98.017|   acoustic|
|       Say Something|A Great Big World...|      74.0|       0.407| 0.147|141.284|   acoustic|
|           I'm Yours|          Jason Mraz|      8

In [12]:
# ordenamos las canciones por popularidad
df_select.orderBy(
    df_select.popularity.desc()
).show(10)

+--------------------+--------------------+----------+------------+------+-------+-----------+
|                song|             artists|popularity|danceability|energy|  tempo|track_genre|
+--------------------+--------------------+----------+------------+------+-------+-----------+
|Unholy (feat. Kim...|Sam Smith;Kim Petras|     100.0|       0.714| 0.472|131.121|      dance|
|Unholy (feat. Kim...|Sam Smith;Kim Petras|     100.0|       0.714| 0.472|131.121|        pop|
|Quevedo: Bzrp Mus...|    Bizarrap;Quevedo|      99.0|       0.621| 0.782|128.033|    hip-hop|
|          La Bachata|       Manuel Turizo|      98.0|       0.835| 0.679| 124.98|      latin|
|     I'm Good (Blue)|David Guetta;Bebe...|      98.0|       0.561| 0.965| 128.04|      dance|
|     I'm Good (Blue)|David Guetta;Bebe...|      98.0|       0.561| 0.965| 128.04|        edm|
|          La Bachata|       Manuel Turizo|      98.0|       0.835| 0.679| 124.98|     latino|
|     I'm Good (Blue)|David Guetta;Bebe...|      9